In [ ]:
import pandas as pd
import numpy as np
import sys
import os

sys.path.append(os.path.abspath(".."))
from disaggregation import disaggregate

### Data source: EU Biomass flows
[EU Biomass flows](http://data.europa.eu/89h/34178536-7fd1-4d5e-b0d4-116be8e4b124) is a dataset that stores harmonised data on biomass supply, uses and flows in the EU. This dataset mirrors the diagrams displayed in the interactive [EU Biomass Flows tool](https://datam.jrc.ec.europa.eu/datam/mashup/BIOMASS_FLOWS/) that represent the flows of biomass for each sector of the bioeconomy, from supply to uses including trade and food waste. The tool enables deeper analysis and comparison of the different countries and sectors across a defined time series. The EU Biomass Flows tool is based on the energy Sankey tool developed by Eurostat.

The dataset can be downloaded as csv file and directly loaded to into the function. Change the `file_path` variable accordingly

In [ ]:
# Default settings

# 1. Load the dataset
file_path = "Dataset_JRC_-_Biomass_uses_and_flows_RAW.csv"
df = pd.read_csv(file_path)

# 2. Define filtering choices
agriculture_filters = {
    # The specific year to filter the agriculture data
    'Year': 2020,
    # The Sankey diagram level to use (e.g., 'L3' for detailed flows)
    'Level_Sankey (Id)': 'L3',
    # The unit is kilotonnes of dry biomass, but flows can be expressed as:
    # Net flow: 'KT_DRY_NET' i.e., difference between import and exports
    # Gross flows: 'KT_DRY_NET' i.e., the total gross amount of flow
    'Unit (Id)': 'KT_DRY_NET',
    # The geographical region to filter by (e.g., 'EU27' for the aggregate EU level)
    'Geopolitical Entity (Id)': 'EU27'
}

fisheries_filters = {
    'Year': 2016,
    'Level_Sankey (Id)': 'L3',
    'Unit (Id)': 'KT_DRY_NET',
    'Geopolitical Entity (Id)': 'EU27',
    'Geopolitical Entity': 'European Union (27 countries)'
}

forestry_filters = {
    'Year': 2017,
    'Level_Sankey (Id)': 'L3',
    'Unit (Id)': 'KT_DRY_NET',
    'Geopolitical Entity (Id)': 'EU27'
}

# 3. Define mappings for agriculture
target_mappings = {
    # Defines how different types of agricultural residues are mapped to their potential end uses
    'residues': {
        'sources': ['Cereals residues', 'Fibre Crops residues', 'Fodder crops residues', 
                   'Fruits residues', 'Oil crops residues', 'Olive trees residues', 
                   'Other industrial crops residues', 'Pulses & protein crops residues', 
                   'Root crops residues', 'Vegetables residues'],
        'targets': ['Biofuels', 'Feed & food products', 'Unknown/Losses']
    },
    # Maps various crop production types to their end uses
    'production': {
        'sources': ['Cereals Production', 'Fodder crops Production', 'Fruits Production',
                   'Oil crops Production', 'Olive trees Production', 'Pulses & protein crops Production',
                   'Root crops Production', 'Vegetables Production'],
        'targets': ['Biofuels', 'Exports', 'Feed & food products', 'Unknown/Losses']
    },
    # Specifies the end uses for grazing
    'grazing': {
        'sources': ['Grazing'],
        'targets': ['Biofuels', 'Exports', 'Feed & food products', 'Fibres and others', 'Unknown/Losses']
    },
    # Defines the end uses for fiber and other industrial crops
    'fiber_other': {
        'sources': ['Fibre Crops Production', 'Other industrial crops Production'],
        'targets': ['Exports', 'Unknown/Losses', 'Fibres and others']
    }
}

import_mappings = {
    # Defines how imported biomass products are mapped to their respective end uses within the system
    'Animal products (feed eq.)': ['Feed & food products', 'Unknown/Losses'],
    'Plant products': ['Exports', 'Feed & food products', 'Fibres and others', 'Unknown/Losses'],
    'Plant-based food': ['Feed & food products', 'Unknown/Losses'],
    'Processed products (biomass eq.)': ['Exports', 'Fibres and others', 'Unknown/Losses']
}

# 4. Define parameters for fisheries
initial_flexibility = 1e-4
min_flexibility = 1e-8
reduction_factor = 0.5

# Define the validity rules for fisheries flows
fisheries_validity_rules = {
    # Specifies valid connections between sources and targets for fisheries flows, ensuring logical consistency in the disaggregation
    # For example, it defines which outputs are possible for 'Capture fisheries'
    'Imports': {
        'Fish & seafood': ['Aquatic-based food', 'Exports', 'Waste'],
        'Fishmeal & oil': ['Fishmeal & oil', 'Waste']
    },
    'Unknown origin': ['Fishmeal & oil', 'Aquatic-based food', 'Waste'],
    'Aquaculture': ['Aquatic-based food', 'Waste'],
    'Capture fisheries': ['Aquatic-based food', 'Exports']
}

# 5. Define allowed links for crops
feed_food_allowed_links = {
    # Defines the allowed connections for disaggregating 'Feed & food products' into more specific categories 
    # like 'Feed & bedding' and 'Plant-based food supply'. This ensures that products are allocated to plausible end uses
    'Animal products (feed eq.)': ['Feed & bedding'],
    'Imports': ['Feed & bedding'],
    'Fishmeal & oil for feed': ['Feed & bedding'],
    'Fodder crops Production': ['Feed & bedding'],
    'Fruits residues': ['Feed & bedding'],
    'Oil crops residues': ['Feed & bedding'],
    'Olive trees residues': ['Feed & bedding'],
    'Other industrial crops residues': ['Feed & bedding'],
    'Root crops residues': ['Feed & bedding'],
    'Cereals Production': ['Feed & bedding', 'Plant-based food supply'],
    'Fruits Production': ['Plant-based food supply'],
    'Grazing': ['Feed & bedding'],
    'Oil crops Production': ['Plant-based food supply'],
    'Olive trees Production': ['Plant-based food supply'],
    'Pulses & protein crops Production': ['Feed & bedding', 'Plant-based food supply'],
    'Root crops Production': ['Plant-based food supply'],
    'Vegetables Production': ['Plant-based food supply'],
    'Fibre Crops residues': ['Feed & bedding'],
    'Fodder crops residues': ['Feed & bedding'],
    'Pulses & protein crops residues': ['Feed & bedding'],
    'Vegetables residues': ['Feed & bedding']
}

# 6. Call the disaggregate function
disaggregated_df = disaggregate(
    df,
    agriculture_filters,
    fisheries_filters,
    forestry_filters,
    target_mappings,
    import_mappings,
    initial_flexibility,
    min_flexibility,
    reduction_factor,
    fisheries_validity_rules,
    feed_food_allowed_links
)

# 7. Save the result to a new CSV file
disaggregated_df.to_csv("Disaggregated_data.csv", index=False)

print("Disaggregation complete. The result has been saved to Disaggregated_data.csv")